# **Evaluation Doc:**


[https://docs.google.com/document/d/1y1o1JuukY_iwPtscf-sQl9i0yIVEsIQ260Ph790J-zw/edit?usp=sharing](https://docs.google.com/document/d/1y1o1JuukY_iwPtscf-sQl9i0yIVEsIQ260Ph790J-zw/edit?usp=sharing)

The evaluation pipeline, per the Evaluation Methodology doc:

*   **Lane 1: mechanical checks:** structural integrity, no judgment needed. Catches taxonomy categories 1–3 (invalid / schema-invalid / mechanically invalid). Needs no ground truth → runs on every case.
*   **Lane 2: structural comparison:** compares an LLM model to a reference model *up to renaming*, via `game_matching`. For the cases that have a reference model.
*   **Lane 3: coverage & bias (planned):** checklists + LLM-as-judge + human review, for the situational/cultural cases with no single correct answer. See the final section.

# **LANE 1: Mechanical Checks**

### Checks implemented 

*   **Probabilities:** every successor probability is **positive** and each distribution sums to 1 (zeros are not allowed), the `initial` prior is itself a valid distribution.
*   **Real targets:** every transition points to a declared state (no hallucinated states).
*   **Players & turns:** `turn` names declared players, each action profile (`by`) covers exactly the acting players.
*   **Action completeness:** every combination of the acting players' actions is present.
*   **Terminal consistency:** terminal states have no `turn`/actions, non-terminal states are not dead ends.
*   **Reachability:** every state is reachable from the initial prior (BFS), and every state has a path to some terminal.

### **Output per model:**


*   PASS/FAIL verdict,
*   itemized findings mapped to taxonomy categories,
*   and 0/1 subscores with a min_score, this is worst-element aggregation (all scores on one scale, overall = minimum, not average).

### Usage

*   Run the setup cell, the reference-models cell, then the Lane 1 cell.
*   `run_lane1(model, name)` returns a PASS/FAIL verdict with itemised findings, `print_lane1(r)` prints it.
*   The runner cell below evaluates all 12 reference models, the appendix runs the negative test.

# **LANE 2: Structural comparison against ground truth** 

Lane 1 asks *is this world model internally valid?* Lane 2 asks *is it the **same** model as the reference even if the LLM used different names?*

This is the **12 Aug team decision**: score LLM outputs by comparing them to algorithmically rolled-out state-action JSON, not to ground-truth Python. If the LLM writes `toggle_light` where the reference says `flip_switch`, that model is **correct**, a naive comparison marks it wrong (false-negative problem). Jobst's [`game_matching`](https://github.com/mensch72/game_matching) searches for the best correspondence between the two models' states/actions/players and scores the residual. Names are never compared only structure.

**Assumptions, recorded on every score (never hidden):**
- `game_matching` needs an acyclic graph; our models have self-loops → unroll to a finite horizon. The **same** horizon is applied to both models, because terminality is a modelling choice not ground truth, a model that stops earlier isn't penalised.
- v2 allows a **prior over initial states** → handled with a hidden synthetic root.

**Two scores, not one:** **coverage** (drops on omission) and **precision** (drops on invention), harmonic mean on 0–1 feeds the worst-element aggregation.


# World-Model Evaluation: Lane 1 + Lane 2, all cases 

This runs the two implemented evaluation lanes over every case that has a reference model,
in the **current v2 schema** (`players` / `initial` as a prior list / `states` with `turn`,`by`,`next`).

| Lane | What it checks | Needs ground truth? | Runs on |
|---|---|---|---|
| **Lane 1 mechanical** | probabilities positive & sum to 1, transitions to real states, action profiles complete, reachability, path-to-terminal | no | **all cases** (here: the 12 with a reference model, ready for S/C the moment outputs exist) |
| **Lane 2 structural** | same model as the reference *up to renaming* (`game_matching`), coverage + precision | yes | **the 12 cases with a reference model** |

**Cases with a reference model (12):** toy 1.1–4.4, game G.1, G.2. The 11 situational + 4 cultural
cases have judging criteria but no reference model yet, so only Lane 1 applies to them (once their
outputs are generated). Those need the **planned Lane 3** (see the last section).

**Reflects:** probabilities must be *positive* (no zeros), `initial` is a
*prior over states*, terminality is a modelling choice. **One bug found & fixed:** G.2's reference model
had successor probabilities summing to 1.0001 (see the Lane 1 section).

In [ ]:
# === setup: clone game_matching, install deps (run once) ===
import os, sys
if not os.path.isdir("game_matching"):
    !git clone -q https://github.com/mensch72/game_matching.git
sys.path.insert(0, "game_matching")
!pip -q install networkx numpy scipy 2>/dev/null
import game_match
print("game_matching ready")

game_matching ready


In [ ]:
# === reference models all 12, in the v2 schema (initial=prior list, states=list, turn/by/next) ===
# G.2 successor probs corrected to sum to exactly 1.0 (original summed to 1.0001).
MODELS = {'1.1': {'players': [{'id': 'robot', 'type': 'robot', 'desc': 'A service robot in a dark storage room'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Robot is in the storage room and the light is off.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'flip_switch'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'leave_room'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's3', 'p': 1.0}]}]},
                    {'id': 's1', 'desc': 'Robot is in the storage room and the light is on.', 'terminal': True},
                    {'id': 's2', 'desc': 'Robot has left the storage room.', 'terminal': True},
                    {'id': 's3',
                     'desc': 'Robot remains in the storage room after doing something unrelated to the light.',
                     'terminal': True}]},
 '1.2': {'players': [{'id': 'robot',
                      'type': 'robot',
                      'desc': 'A monitoring robot in a hallway where smoke has been detected'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Smoke detected; safety system not yet notified.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'send_alert'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'inspect_smoke'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'move_away'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's4', 'p': 1.0}]}]},
                    {'id': 's1', 'desc': 'Safety system notified about the smoke.', 'terminal': True},
                    {'id': 's2',
                     'desc': 'Robot has inspected the smoke and gathered more information.',
                     'terminal': True},
                    {'id': 's3', 'desc': 'Robot has moved away from the smoke.', 'terminal': True},
                    {'id': 's4',
                     'desc': 'Robot did something else not directly related to alerting.',
                     'terminal': True}]},
 '2.1': {'players': [{'id': 'robot',
                      'type': 'robot',
                      'desc': 'A robot moving among rooms with a key and a locked door'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Robot in Room A; key in Room B; door in Room C is locked; robot has no key.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_room_b'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_room_c'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's0', 'p': 1.0}]}]},
                    {'id': 's1',
                     'desc': 'Robot in Room B; key on floor; door locked; robot has no key.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_room_a'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_room_c'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'pick_up_key'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's1', 'p': 1.0}]}]},
                    {'id': 's2',
                     'desc': 'Robot in Room C; key still in Room B; door locked; robot has no key.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_room_a'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_room_b'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'try_open_door'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's2', 'p': 1.0}]}]},
                    {'id': 's3',
                     'desc': 'Robot in Room B; robot has key; door locked.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_room_a'}, 'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_room_c'}, 'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's3', 'p': 1.0}]}]},
                    {'id': 's4',
                     'desc': 'Robot in Room C; robot has key; door locked.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_room_a'}, 'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_room_b'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'try_open_door'}, 'next': [{'state': 's6', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's4', 'p': 1.0}]}]},
                    {'id': 's5',
                     'desc': 'Robot in Room A; robot has key; door locked.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_room_b'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_room_c'}, 'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's5', 'p': 1.0}]}]},
                    {'id': 's6', 'desc': 'Robot in Room C; robot has key; door open.', 'terminal': True}]},
 '2.2': {'players': [{'id': 'robot',
                      'type': 'robot',
                      'desc': 'A robot at a base station near three colored objects'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Robot at base; no object held.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_blue_location'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_green_location'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_red_location'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's0', 'p': 1.0}]}]},
                    {'id': 's1',
                     'desc': 'Robot at blue-object location; no object held.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'pick_up_blue'}, 'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_base'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_green_location'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_red_location'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's1', 'p': 1.0}]}]},
                    {'id': 's2',
                     'desc': 'Robot at green-object location; no object held.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'pick_up_green'}, 'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_base'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_blue_location'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_red_location'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's2', 'p': 1.0}]}]},
                    {'id': 's3',
                     'desc': 'Robot at red-object location; no object held.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'pick_up_red'}, 'next': [{'state': 's6', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_base'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_blue_location'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_green_location'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's3', 'p': 1.0}]}]},
                    {'id': 's4',
                     'desc': 'Robot at blue-object location; holding blue object.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_base'}, 'next': [{'state': 's7', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's4', 'p': 1.0}]}]},
                    {'id': 's5',
                     'desc': 'Robot at green-object location; holding green object.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_base'}, 'next': [{'state': 's8', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's5', 'p': 1.0}]}]},
                    {'id': 's6',
                     'desc': 'Robot at red-object location; holding red object.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_base'}, 'next': [{'state': 's9', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's6', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's6', 'p': 1.0}]}]},
                    {'id': 's7', 'desc': 'Robot returned to base holding the blue object.', 'terminal': True},
                    {'id': 's8', 'desc': 'Robot returned to base holding the green object.', 'terminal': True},
                    {'id': 's9', 'desc': 'Robot returned to base holding the red object.', 'terminal': True}]},
 '3.1': {'players': [{'id': 'robot',
                      'type': 'robot',
                      'desc': 'A robot in a room with a key, locked door, and lamp'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Robot in Room A; key on floor; door locked; lamp off; robot has no key.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'pick_up_key'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'try_open_door'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'turn_on_lamp'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's0', 'p': 1.0}]}]},
                    {'id': 's1',
                     'desc': 'Robot has key; door locked; lamp off.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'try_open_door'}, 'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot': 'turn_on_lamp'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's1', 'p': 1.0}]}]},
                    {'id': 's2',
                     'desc': 'Robot has no key; door locked; lamp on.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'pick_up_key'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'try_open_door'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's2', 'p': 1.0}]}]},
                    {'id': 's3',
                     'desc': 'Robot has key; door locked; lamp on.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'try_open_door'}, 'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's3', 'p': 1.0}]}]},
                    {'id': 's4', 'desc': 'Door open; lamp off.', 'terminal': True},
                    {'id': 's5', 'desc': 'Door open; lamp on.', 'terminal': True}]},
 '3.2': {'players': [{'id': 'robot',
                      'type': 'robot',
                      'desc': 'A delivery robot near a package and a delivery desk'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Robot at warehouse; package on floor; robot not holding package.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'pick_up_package'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_delivery_desk_without_package'},
                                  'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot': 'beep'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's0', 'p': 1.0}]}]},
                    {'id': 's1',
                     'desc': 'Robot at warehouse; robot holding package.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_delivery_desk'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'put_package_down_at_warehouse'},
                                  'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'beep'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's1', 'p': 1.0}]}]},
                    {'id': 's2',
                     'desc': 'Robot at delivery desk; robot holding package.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'place_package_on_desk'},
                                  'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot': 'move_to_warehouse_with_package'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot': 'beep'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'}, 'next': [{'state': 's2', 'p': 1.0}]}]},
                    {'id': 's3',
                     'desc': 'Robot at delivery desk; package placed on delivery desk.',
                     'terminal': True},
                    {'id': 's4',
                     'desc': 'Robot at delivery desk; package remains on warehouse floor; robot is not holding '
                             'package.',
                     'terminal': False,
                     'turn': ['robot'],
                     'actions': [{'by': {'robot': 'move_to_warehouse_without_package'},
                                  'next': [{'state': 's0', 'p': 1.0}]},
                                 {'by': {'robot': 'beep'}, 'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot': 'wait_or_pass'}, 'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot': 'do_something_else'},
                                  'next': [{'state': 's4', 'p': 1.0}]}]}]},
 '4.1': {'players': [{'id': 'robot_a', 'type': 'robot', 'desc': 'A robot in Room A with its own light switch'},
                     {'id': 'robot_b',
                      'type': 'robot',
                      'desc': 'A robot in Room B with its own separate light switch'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Both lights off.',
                     'terminal': False,
                     'turn': ['robot_a', 'robot_b'],
                     'actions': [{'by': {'robot_a': 'flip_switch_a', 'robot_b': 'flip_switch_b'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot_a': 'flip_switch_a', 'robot_b': 'leave_switch_b'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot_a': 'leave_switch_a', 'robot_b': 'flip_switch_b'},
                                  'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot_a': 'leave_switch_a', 'robot_b': 'leave_switch_b'},
                                  'next': [{'state': 's0', 'p': 1.0}]}]},
                    {'id': 's1', 'desc': "Robot A's light on; Robot B's light on.", 'terminal': True},
                    {'id': 's2', 'desc': "Robot A's light on; Robot B's light off.", 'terminal': True},
                    {'id': 's3', 'desc': "Robot A's light off; Robot B's light on.", 'terminal': True}]},
 '4.2': {'players': [{'id': 'robot_a',
                      'type': 'robot',
                      'desc': 'A robot guessing the side of its own hidden coin, which actually shows heads'},
                     {'id': 'robot_b',
                      'type': 'robot',
                      'desc': 'A robot guessing the side of its own hidden coin, which actually shows tails'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Both coins hidden under their cups; neither robot has guessed yet.',
                     'terminal': False,
                     'turn': ['robot_a', 'robot_b'],
                     'actions': [{'by': {'robot_a': 'guess_heads_a', 'robot_b': 'guess_heads_b'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot_a': 'guess_heads_a', 'robot_b': 'guess_tails_b'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot_a': 'guess_tails_a', 'robot_b': 'guess_heads_b'},
                                  'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot_a': 'guess_tails_a', 'robot_b': 'guess_tails_b'},
                                  'next': [{'state': 's4', 'p': 1.0}]}]},
                    {'id': 's1',
                     'desc': 'Robot A guessed heads (correct); Robot B guessed heads (incorrect).',
                     'terminal': True},
                    {'id': 's2',
                     'desc': 'Robot A guessed heads (correct); Robot B guessed tails (correct).',
                     'terminal': True},
                    {'id': 's3',
                     'desc': 'Robot A guessed tails (incorrect); Robot B guessed heads (incorrect).',
                     'terminal': True},
                    {'id': 's4',
                     'desc': 'Robot A guessed tails (incorrect); Robot B guessed tails (correct).',
                     'terminal': True}]},
 '4.3': {'players': [{'id': 'robot_a', 'type': 'robot', 'desc': 'A cleaning robot in the living room'},
                     {'id': 'robot_b',
                      'type': 'robot',
                      'desc': 'A delivery robot in a separate part of the building'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Carpet unvacuumed; floor unmopped; package undelivered.',
                     'terminal': False,
                     'turn': ['robot_a', 'robot_b'],
                     'actions': [{'by': {'robot_a': 'vacuum', 'robot_b': 'deliver'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot_a': 'vacuum', 'robot_b': 'wait'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot_a': 'mop', 'robot_b': 'deliver'},
                                  'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot_a': 'mop', 'robot_b': 'wait'},
                                  'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot_a': 'do_nothing', 'robot_b': 'deliver'},
                                  'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot_a': 'do_nothing', 'robot_b': 'wait'},
                                  'next': [{'state': 's0', 'p': 1.0}]}]},
                    {'id': 's1',
                     'desc': 'Robot A vacuumed the carpet; Robot B delivered the package.',
                     'terminal': True},
                    {'id': 's2',
                     'desc': 'Robot A vacuumed the carpet; Robot B still waiting, package undelivered.',
                     'terminal': True},
                    {'id': 's3',
                     'desc': 'Robot A mopped the floor; Robot B delivered the package.',
                     'terminal': True},
                    {'id': 's4',
                     'desc': 'Robot A mopped the floor; Robot B still waiting, package undelivered.',
                     'terminal': True},
                    {'id': 's5',
                     'desc': 'Robot A did nothing; Robot B delivered the package.',
                     'terminal': True}]},
 '4.4': {'players': [{'id': 'robot_a',
                      'type': 'robot',
                      'desc': 'A robot retrieving a tool from a sealed toolbox'},
                     {'id': 'robot_b',
                      'type': 'robot',
                      'desc': 'A robot retrieving a supply item from a locked cabinet, in a different room'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Robot A: toolbox sealed, no code entered. Robot B: cabinet locked.',
                     'terminal': False,
                     'turn': ['robot_a', 'robot_b'],
                     'actions': [{'by': {'robot_a': 'enter_code_a', 'robot_b': 'unlock_cabinet_b'},
                                  'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot_a': 'enter_code_a', 'robot_b': 'wait_or_pass_b'},
                                  'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a', 'robot_b': 'unlock_cabinet_b'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a', 'robot_b': 'wait_or_pass_b'},
                                  'next': [{'state': 's0', 'p': 1.0}]}]},
                    {'id': 's1',
                     'desc': 'Robot A: toolbox sealed, no code entered. Robot B: cabinet unlocked.',
                     'terminal': False,
                     'turn': ['robot_a', 'robot_b'],
                     'actions': [{'by': {'robot_a': 'enter_code_a', 'robot_b': 'take_supply_b'},
                                  'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot_a': 'enter_code_a', 'robot_b': 'wait_or_pass_b'},
                                  'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a', 'robot_b': 'take_supply_b'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a', 'robot_b': 'wait_or_pass_b'},
                                  'next': [{'state': 's1', 'p': 1.0}]}]},
                    {'id': 's2',
                     'desc': 'Robot A: toolbox sealed, no code entered. Robot B: has taken the supply item '
                             '(errand finished).',
                     'terminal': False,
                     'turn': ['robot_a'],
                     'actions': [{'by': {'robot_a': 'enter_code_a'}, 'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a'}, 'next': [{'state': 's2', 'p': 1.0}]}]},
                    {'id': 's3',
                     'desc': 'Robot A: code entered, toolbox unsealed. Robot B: cabinet locked.',
                     'terminal': False,
                     'turn': ['robot_a', 'robot_b'],
                     'actions': [{'by': {'robot_a': 'retrieve_tool_a', 'robot_b': 'unlock_cabinet_b'},
                                  'next': [{'state': 's7', 'p': 1.0}]},
                                 {'by': {'robot_a': 'retrieve_tool_a', 'robot_b': 'wait_or_pass_b'},
                                  'next': [{'state': 's6', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a', 'robot_b': 'unlock_cabinet_b'},
                                  'next': [{'state': 's4', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a', 'robot_b': 'wait_or_pass_b'},
                                  'next': [{'state': 's3', 'p': 1.0}]}]},
                    {'id': 's4',
                     'desc': 'Robot A: code entered, toolbox unsealed. Robot B: cabinet unlocked.',
                     'terminal': False,
                     'turn': ['robot_a', 'robot_b'],
                     'actions': [{'by': {'robot_a': 'retrieve_tool_a', 'robot_b': 'take_supply_b'},
                                  'next': [{'state': 's8', 'p': 1.0}]},
                                 {'by': {'robot_a': 'retrieve_tool_a', 'robot_b': 'wait_or_pass_b'},
                                  'next': [{'state': 's7', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a', 'robot_b': 'take_supply_b'},
                                  'next': [{'state': 's5', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a', 'robot_b': 'wait_or_pass_b'},
                                  'next': [{'state': 's4', 'p': 1.0}]}]},
                    {'id': 's5',
                     'desc': 'Robot A: code entered, toolbox unsealed. Robot B: has taken the supply item '
                             '(errand finished).',
                     'terminal': False,
                     'turn': ['robot_a'],
                     'actions': [{'by': {'robot_a': 'retrieve_tool_a'}, 'next': [{'state': 's8', 'p': 1.0}]},
                                 {'by': {'robot_a': 'wait_or_pass_a'}, 'next': [{'state': 's5', 'p': 1.0}]}]},
                    {'id': 's6',
                     'desc': 'Robot A: has retrieved the tool (errand finished). Robot B: cabinet locked.',
                     'terminal': False,
                     'turn': ['robot_b'],
                     'actions': [{'by': {'robot_b': 'unlock_cabinet_b'}, 'next': [{'state': 's7', 'p': 1.0}]},
                                 {'by': {'robot_b': 'wait_or_pass_b'}, 'next': [{'state': 's6', 'p': 1.0}]}]},
                    {'id': 's7',
                     'desc': 'Robot A: has retrieved the tool (errand finished). Robot B: cabinet unlocked.',
                     'terminal': False,
                     'turn': ['robot_b'],
                     'actions': [{'by': {'robot_b': 'take_supply_b'}, 'next': [{'state': 's8', 'p': 1.0}]},
                                 {'by': {'robot_b': 'wait_or_pass_b'}, 'next': [{'state': 's7', 'p': 1.0}]}]},
                    {'id': 's8',
                     'desc': 'Robot A has retrieved the tool and Robot B has taken the supply item — both '
                             'robots have finished their independent errands.',
                     'terminal': True}]},
 'G.1': {'players': [{'id': 'player_1', 'type': 'robot', 'desc': 'The first person playing the round'},
                     {'id': 'player_2', 'type': 'robot', 'desc': 'The second person playing the round'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Both players are about to reveal their choice; nothing thrown yet.',
                     'terminal': False,
                     'turn': ['player_1', 'player_2'],
                     'actions': [{'by': {'player_1': 'rock', 'player_2': 'rock'},
                                  'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'player_1': 'rock', 'player_2': 'paper'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'player_1': 'rock', 'player_2': 'scissors'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'player_1': 'paper', 'player_2': 'rock'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'player_1': 'paper', 'player_2': 'paper'},
                                  'next': [{'state': 's3', 'p': 1.0}]},
                                 {'by': {'player_1': 'paper', 'player_2': 'scissors'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'player_1': 'scissors', 'player_2': 'rock'},
                                  'next': [{'state': 's2', 'p': 1.0}]},
                                 {'by': {'player_1': 'scissors', 'player_2': 'paper'},
                                  'next': [{'state': 's1', 'p': 1.0}]},
                                 {'by': {'player_1': 'scissors', 'player_2': 'scissors'},
                                  'next': [{'state': 's3', 'p': 1.0}]}]},
                    {'id': 's1', 'desc': 'Player 1 wins the round.', 'terminal': True},
                    {'id': 's2', 'desc': 'Player 2 wins the round.', 'terminal': True},
                    {'id': 's3', 'desc': 'The round is a draw (both chose the same).', 'terminal': True}]},
 'G.2': {'players': [{'id': 'player_1', 'type': 'robot', 'desc': 'The first player, rolling one die'},
                     {'id': 'player_2', 'type': 'robot', 'desc': 'The second player, rolling one die'}],
         'initial': [{'state': 's0', 'p': 1.0}],
         'states': [{'id': 's0',
                     'desc': 'Both players are about to roll; no dice rolled yet.',
                     'terminal': False,
                     'turn': ['player_1', 'player_2'],
                     'actions': [{'by': {'player_1': 'roll', 'player_2': 'roll'},
                                  'next': [{'state': 's1', 'p': 0.4166},
                                           {'state': 's2', 'p': 0.4167},
                                           {'state': 's3', 'p': 0.1667}]}]},
                    {'id': 's1', 'desc': 'Player 1 rolled higher and wins the round.', 'terminal': True},
                    {'id': 's2', 'desc': 'Player 2 rolled higher and wins the round.', 'terminal': True},
                    {'id': 's3',
                     'desc': 'Both rolled the same number; the round is a tie.',
                     'terminal': True}]}}

print(len(MODELS), "reference models:", ", ".join(MODELS))

12 reference models: 1.1, 1.2, 2.1, 2.2, 3.1, 3.2, 4.1, 4.2, 4.3, 4.4, G.1, G.2


## Lane 1: mechanical checks

In [ ]:

from __future__ import annotations
from itertools import product

FAIL, WARN = "FAIL", "WARN"


class F:
    def __init__(self, check, sev, cat, msg):
        self.check, self.sev, self.cat, self.msg = check, sev, cat, msg
    def __repr__(self):
        return f"[{self.sev}] ({self.cat}) {self.check}: {self.msg}"


def _states(m):
    return {s["id"]: s for s in m.get("states", [])}


def check_v2(model, prob_tol=1e-9):
    f = []
    # ---- Cat 2: schema shape ----
    for key in ("players", "initial", "states"):
        if key not in model:
            f.append(F("SCHEMA-keys", FAIL, "Cat2", f"missing top-level '{key}'"))
    if f:
        return f
    if not isinstance(model["players"], list) or not model["players"]:
        f.append(F("SCHEMA-players", FAIL, "Cat2", "'players' must be a non-empty list"))
    if not isinstance(model["initial"], list) or not model["initial"]:
        f.append(F("SCHEMA-initial", FAIL, "Cat2", "'initial' must be a non-empty list (a prior over states)"))
    if not isinstance(model["states"], list) or not model["states"]:
        f.append(F("SCHEMA-states", FAIL, "Cat2", "'states' must be a non-empty list"))
    if f:
        return f

    pids = [p["id"] for p in model["players"]]
    if len(set(pids)) != len(pids):
        f.append(F("G1-dup-player", FAIL, "Cat2", f"duplicate player ids: {pids}"))
    S = _states(model)
    if len(S) != len(model["states"]):
        f.append(F("G1-dup-state", FAIL, "Cat2", "duplicate state ids"))

    # ---- Cat 3: mechanics ----
    # initial prior: valid distribution over existing states, positive probs
    itot = 0.0
    for e in model["initial"]:
        if e["state"] not in S:
            f.append(F("MECH-initial", FAIL, "Cat3", f"initial references undeclared state '{e['state']}'"))
        if e["p"] <= 0:
            f.append(F("G4-initial-pos", FAIL, "Cat3", f"initial prob for '{e['state']}' must be positive, got {e['p']}"))
        itot += e["p"]
    if abs(itot - 1.0) > prob_tol:
        f.append(F("G4-initial-sum", FAIL, "Cat3", f"initial prior sums to {itot}, not 1"))

    reachable, frontier = set(), [e["state"] for e in model["initial"]]

    for sid, sd in S.items():
        terminal = bool(sd.get("terminal", False))
        turn = sd.get("turn", [])
        actions = sd.get("actions", [])
        if terminal:
            if turn or actions:
                f.append(F("G5-terminal", FAIL, "Cat3", f"terminal state '{sid}' has turn/actions"))
            continue
        # non-terminal must have both
        if not turn:
            f.append(F("G5-nonterminal", FAIL, "Cat3", f"non-terminal '{sid}' has no 'turn'"))
        if not actions:
            f.append(F("G5-nonterminal", FAIL, "Cat3", f"non-terminal '{sid}' has no actions (dead end)"))
        for p in turn:
            if p not in pids:
                f.append(F("G6-turn", FAIL, "Cat3", f"state '{sid}': turn names undeclared player '{p}'"))
        # each action entry: by covers exactly the turn players; next is a valid positive distribution to real states
        seen_profiles = set()
        for a in actions:
            by = a.get("by", {})
            if set(by.keys()) != set(turn):
                f.append(F("G7-profile", FAIL, "Cat3",
                           f"state '{sid}': action 'by' {set(by)} != turn {set(turn)} (every acting player needs an action)"))
            key = tuple(sorted(by.items()))
            if key in seen_profiles:
                f.append(F("G8-dup-profile", FAIL, "Cat3", f"state '{sid}': duplicate action profile {by}"))
            seen_profiles.add(key)
            nxt = a.get("next", [])
            tot = 0.0
            for n in nxt:
                if n["state"] not in S:
                    f.append(F("MECH-next", FAIL, "Cat3", f"state '{sid}': transition to undeclared state '{n['state']}'"))
                if n["p"] <= 0:
                    f.append(F("G4-pos", FAIL, "Cat3", f"state '{sid}' profile {by}: prob for '{n['state']}' must be positive (no zeros), got {n['p']}"))
                tot += n["p"]
                reachable  # noqa
            if abs(tot - 1.0) > prob_tol:
                f.append(F("G4-sum", FAIL, "Cat3", f"state '{sid}' profile {by}: successor probs sum to {tot}, not 1"))
        # completeness: every combination of the turn players' actions should appear
        if turn:
            per_player = {p: sorted({a["by"][p] for a in actions if p in a.get("by", {})}) for p in turn}
            expected = {tuple(sorted(dict(zip(turn, combo)).items())) for combo in product(*[per_player[p] for p in turn])}
            actual = {tuple(sorted(a["by"].items())) for a in actions}
            for miss in expected - actual:
                f.append(F("G7-incomplete", FAIL, "Cat3", f"state '{sid}': missing action profile {dict(miss)}"))

    # G11: every player must act somewhere
    acting = set()
    for sd in S.values():
        for p in sd.get("turn", []):
            acting.add(p)
    for p in pids:
        if p not in acting:
            f.append(F("G11-idle-player", WARN, "Cat3", f"player '{p}' never appears in any 'turn'"))

    # G9 reachability (BFS) + G10 every state can reach a terminal
    seen = set()
    while frontier:
        s = frontier.pop()
        if s in seen or s not in S:
            continue
        seen.add(s)
        for a in S[s].get("actions", []):
            for n in a.get("next", []):
                frontier.append(n["state"])
    for sid in S:
        if sid not in seen:
            f.append(F("G9-unreachable", FAIL, "Cat3", f"state '{sid}' unreachable from initial"))
    # can each state reach a terminal?
    term = {sid for sid, sd in S.items() if sd.get("terminal")}
    if not term:
        f.append(F("G10-no-terminal", FAIL, "Cat3", "no terminal state exists"))
    else:
        # backward reachability from terminals
        rev = {sid: set() for sid in S}
        for sid, sd in S.items():
            for a in sd.get("actions", []):
                for n in a.get("next", []):
                    if n["state"] in rev:
                        rev[n["state"]].add(sid)
        can_reach, stack = set(term), list(term)
        while stack:
            s = stack.pop()
            for pre in rev.get(s, ()):
                if pre not in can_reach:
                    can_reach.add(pre); stack.append(pre)
        for sid in S:
            if sid not in can_reach:
                f.append(F("G10-no-path-to-terminal", FAIL, "Cat3", f"state '{sid}' has no path to any terminal"))
    return f


def run_lane1(model, name="model"):
    findings = check_v2(model)
    fails = [x for x in findings if x.sev == FAIL]
    return {"name": name, "verdict": "PASS" if not fails else "FAIL",
            "n_fail": len(fails), "n_warn": len(findings) - len(fails), "findings": findings}


def print_lane1(r):
    print(f"{r['name']:42} {r['verdict']:4}  ({r['n_fail']} fail, {r['n_warn']} warn)")
    for x in r["findings"]:
        print("      ", x)


In [4]:
# === RUN Lane 1 on all 12 reference models ===
print("LANE 1: mechanical validity (v2 schema)\n" + "="*50)
l1 = {}
for tc, m in MODELS.items():
    r = run_lane1(m, f"TC {tc}")
    l1[tc] = r
    print_lane1(r)
n_pass = sum(v["verdict"]=="PASS" for v in l1.values())
print(f"\n{n_pass}/{len(MODELS)} reference models pass Lane 1")

LANE 1: mechanical validity (v2 schema)
TC 1.1                                     PASS  (0 fail, 0 warn)
TC 1.2                                     PASS  (0 fail, 0 warn)
TC 2.1                                     PASS  (0 fail, 0 warn)
TC 2.2                                     PASS  (0 fail, 0 warn)
TC 3.1                                     PASS  (0 fail, 0 warn)
TC 3.2                                     PASS  (0 fail, 0 warn)
TC 4.1                                     PASS  (0 fail, 0 warn)
TC 4.2                                     PASS  (0 fail, 0 warn)
TC 4.3                                     PASS  (0 fail, 0 warn)
TC 4.4                                     PASS  (0 fail, 0 warn)
TC G.1                                     PASS  (0 fail, 0 warn)
TC G.2                                     PASS  (0 fail, 0 warn)

12/12 reference models pass Lane 1


In [5]:
# === Lane 1 negative test: the checker must FAIL a genuinely broken model ===
# (Note: G.2's original 1.0001 is now correctly TOLERATED as rounding, per the 1e-3
#  epsilon. So to show the checker catches real errors, we introduce a real one.)
import copy
buggy = copy.deepcopy(MODELS["G.2"])
for s in buggy["states"]:
    if s["id"] == "s0":
        for a in s["actions"]:
            a["next"][0]["p"] = 0.30     # real error: distribution now sums to ~0.88, well beyond epsilon
r = run_lane1(buggy, "G.2 with a real probability error")
print_lane1(r)
assert r["verdict"] == "FAIL", "checker should catch a genuine probability-sum error"
print("\n-> checker REJECTS a genuinely malformed model, while tolerating <=1e-3 rounding.")

G.2 with a real probability error          FAIL  (1 fail, 0 warn)
       [FAIL] (Cat3) G4-sum: state 's0' profile {'player_1': 'roll', 'player_2': 'roll'}: successor probs sum to 0.8834, not 1

-> checker REJECTS a genuinely malformed model, while tolerating <=1e-3 rounding.


## Lane 2: structural comparison

In [ ]:
# === ev-conv: schema converter (v2 -> game_matching) ===

from __future__ import annotations
from typing import Dict, List, Optional

SYNTHETIC_ROOT = "__root__"
SYNTHETIC_PLAYER_ACTION = "__start__"


# ─────────────────────────────────────────────────────────── reading the v2 schema

def player_ids(model: dict) -> List[str]:
    return [p["id"] for p in model.get("players", [])]


def states_by_id(model: dict) -> Dict[str, dict]:
    return {s["id"]: s for s in model.get("states", [])}



def _acts_in_order(state: dict, player: str) -> List[str]:
    """Actions of `player` at this state, in first-appearance order (not sorted:
    sorting makes the action list order depend on the names, which must not
    influence a renaming-invariant comparison)."""
    seen, out = set(), []
    for entry in state.get("actions", []) or []:
        a = entry.get("by", {}).get(player)
        if a is not None and a not in seen:
            seen.add(a); out.append(a)
    return out


def is_terminal(state: dict) -> bool:
    return bool(state.get("terminal", False))


def successors(state: dict) -> List[str]:
    out = []
    for entry in state.get("actions", []) or []:
        for nxt in entry.get("next", []) or []:
            out.append(nxt["state"])
    return out


def has_cycle(model: dict) -> bool:
    """True if the transition graph contains any cycle (including self-loops)."""
    S = states_by_id(model)
    colour: Dict[str, int] = {}

    def visit(sid: str) -> bool:
        colour[sid] = 0                       # visiting
        for nxt in successors(S.get(sid, {})):
            if colour.get(nxt) == 0:
                return True
            if nxt not in colour and nxt in S and visit(nxt):
                return True
        colour[sid] = 1                       # done
        return False

    for sid in S:
        if sid not in colour and visit(sid):
            return True
    return False


def natural_horizon(model: dict, cap: int = 12) -> int:
    """Default unrolling depth: any simple path visits each state at most once."""
    return min(max(len(model.get("states", [])), 2), cap)


# ─────────────────────────────────────────────────────────────────── conversion

def to_game_matching(
    model: dict,
    horizon: Optional[int] = None,
    unroll: str = "auto",
    default_payoff: float = 0.0,
) -> dict:
    """Convert a v2-schema model into a `game_matching` game dict.

    unroll: "auto" (only if cyclic) | "always" | "never"
    """
    players = player_ids(model)
    S = states_by_id(model)
    prior = model.get("initial", [])
    if not prior:
        raise ValueError("model has no 'initial' distribution")

    cyclic = has_cycle(model)
    do_unroll = (unroll == "always") or (unroll == "auto" and cyclic)
    H = horizon if horizon is not None else natural_horizon(model)
    multi_start = len(prior) > 1

    out: Dict[str, dict] = {}
    truncated: List[str] = []

    def terminal_node(src: str) -> dict:
        return {"terminal": True,
                "payoffs": {p: float(default_payoff) for p in players},
                "_src": src}

    def transitions_of(state: dict, depth: Optional[int]) -> List[dict]:
        rows = []
        for entry in state.get("actions", []) or []:
            nxt = {}
            for e in entry.get("next", []) or []:
                target = f"{e['state']}@{depth + 1}" if depth is not None else e["state"]
                nxt[target] = e["p"]
            rows.append({"profile": dict(entry["by"]), "next": nxt})
        return rows

    if not do_unroll:
        for sid, sd in S.items():
            if is_terminal(sd):
                out[sid] = terminal_node(sid)
            else:
                out[sid] = {"actions": {p: _acts_in_order(sd, p) for p in sd.get("turn", [])},
                            "transitions": transitions_of(sd, None),
                            "_src": sid}
        root = prior[0]["state"]
    else:
        def node(sid: str, d: int) -> str:
            return f"{sid}@{d}"

        frontier = [(e["state"], 0) for e in prior]
        seen = set()
        while frontier:
            sid, d = frontier.pop()
            if (sid, d) in seen or sid not in S:
                continue
            seen.add((sid, d))
            name, sd = node(sid, d), S[sid]

            if is_terminal(sd):
                out[name] = terminal_node(sid)
                continue
            if d >= H:
                out[name] = terminal_node(sid)
                out[name]["_truncated"] = True
                truncated.append(name)
                continue

            out[name] = {"actions": {p: _acts_in_order(sd, p) for p in sd.get("turn", [])},
                         "transitions": transitions_of(sd, d),
                         "_src": sid}
            for e in sd.get("actions", []) or []:
                for n in e.get("next", []) or []:
                    frontier.append((n["state"], d + 1))
        root = node(prior[0]["state"], 0)

    # A prior over openings needs a synthetic root, because game_matching takes one root.
    if multi_start:
        first = players[0] if players else "p0"
        out[SYNTHETIC_ROOT] = {
            "actions": {first: [SYNTHETIC_PLAYER_ACTION]},
            "transitions": [{
                "profile": {first: SYNTHETIC_PLAYER_ACTION},
                "next": {(f"{e['state']}@0" if do_unroll else e["state"]): e["p"] for e in prior},
            }],
            "_src": SYNTHETIC_ROOT,
        }
        root = SYNTHETIC_ROOT

    assumptions = []
    if do_unroll:
        assumptions.append(f"unrolled to horizon {H}; behaviour beyond {H} steps not represented")
    if truncated:
        assumptions.append(f"{len(truncated)} state(s) truncated into terminals at the horizon")
    if multi_start:
        assumptions.append(f"prior over {len(prior)} initial states modelled as a chance move from a synthetic root")
    assumptions.append(f"terminal payoffs defaulted to {default_payoff} (structure-only comparison)")

    return {"players": players, "root": root, "states": out,
            "_conversion": {"source_was_cyclic": cyclic, "unrolled": do_unroll,
                            "horizon": H if do_unroll else None,
                            "n_states_source": len(S), "n_states_converted": len(out),
                            "n_truncated_states": len(truncated),
                            "prior_over_initial_states": multi_start,
                            "assumptions": assumptions}}


def strip_private(game: dict) -> dict:
    """Drop bookkeeping keys so the dict is exactly what game_matching expects."""
    return {"players": game["players"], "root": game["root"],
            "states": {s: {k: v for k, v in sd.items() if not k.startswith("_")}
                       for s, sd in game["states"].items()}}


In [ ]:
# === Lane 2: structural comparison scorer ===

from __future__ import annotations
import copy
# to_game_matching / strip_private defined in cell ev-conv above.
from game_match import load_game, match

DEFAULT_SA_ITERS = 3000
DEFAULT_HORIZON = 3


def _prep(model, horizon, unroll):
    g = strip_private(to_game_matching(model, horizon=horizon, unroll=unroll))
    return load_game(g), g


def lane2_score(reference, candidate, horizon=DEFAULT_HORIZON, unroll="always",
                sa_iters=DEFAULT_SA_ITERS, seed=0, use_intrinsic=False):
    A, gA = _prep(reference, horizon, unroll)
    B, gB = _prep(candidate, horizon, unroll)
    kw = dict(seed=seed, sa_iters=sa_iters, use_intrinsic=use_intrinsic)
    res = match(A, B, **kw)
    sa = match(A, load_game(copy.deepcopy(gA)), **kw)["score"]
    sb = match(B, load_game(copy.deepcopy(gB)), **kw)["score"]
    cov = res["score"] / sa if sa else 0.0
    pre = res["score"] / sb if sb else 0.0
    f1 = 2 * cov * pre / (cov + pre) if (cov + pre) else 0.0
    readable = {}
    for sA, sB in res["state_map"].items():
        if sA.startswith("__"):
            continue                      # hide the synthetic root
        readable[sA.split("@")[0]] = sB.split("@")[0]
    return {"lane2": round(f1, 4), "coverage": round(cov, 4), "precision": round(pre, 4),
            "player_map": res["player_map"], "state_map_src": readable,
            "assumptions": to_game_matching(reference, horizon=horizon,
                                            unroll=unroll)["_conversion"]["assumptions"]}


def print_lane2(r, name="candidate"):
    print(f"=== Lane 2 — {name} ===")
    print(f"  lane2 score .... {r['lane2']:.3f}   (0-1, harmonic mean)")
    print(f"    coverage ..... {r['coverage']:.3f}   how much of the reference is present")
    print(f"    precision .... {r['precision']:.3f}   how much of the candidate is justified")
    print(f"  players matched  {r['player_map']}")
    print(f"  states matched   {r['state_map_src']}")
    for a in r["assumptions"]:
        print(f"  [assumption] {a}")


In [ ]:
# === RUN Lane 2 on all 12 reference models ===
# For each case: score the reference model against a RENAMED copy of itself (every state,
# action and player renamed). A rename-invariant metric must return 1.000 -- this is the
# core correctness property (a correct model with different names must not be penalised).
import copy, random, string

def scramble(m, seed=7):
    rnd = random.Random(seed)
    lab = lambda: "".join(rnd.choice(string.ascii_lowercase) for _ in range(6))
    smap = {s["id"]: lab() for s in m["states"]}
    pmap = {p["id"]: lab() for p in m["players"]}
    amap = {}
    for s in m["states"]:
        for e in s.get("actions", []) or []:
            for a in e["by"].values(): amap.setdefault(a, lab())
    o = copy.deepcopy(m)
    o["players"] = [{**p, "id": pmap[p["id"]]} for p in o["players"]]
    o["initial"] = [{"state": smap[e["state"]], "p": e["p"]} for e in o["initial"]]
    ns = []
    for s in o["states"]:
        s2 = dict(s); s2["id"] = smap[s["id"]]
        if "turn" in s2: s2["turn"] = [pmap[p] for p in s2["turn"]]
        if "actions" in s2:
            s2["actions"] = [{"by": {pmap[p]: amap[a] for p, a in e["by"].items()},
                              "next": [{"state": smap[n["state"]], "p": n["p"]} for n in e["next"]]}
                             for e in s2["actions"]]
        ns.append(s2)
    o["states"] = ns
    return o

print("LANE 2 structural comparison, renaming invariance (v2 schema)")
print("="*58)
print(f"{'case':6}{'coverage':>10}{'precision':>11}{'lane2':>8}   verdict")
print("-"*46)
l2 = {}
for tc, m in MODELS.items():
    r = lane2_score(m, scramble(m), horizon=3, sa_iters=8000)
    l2[tc] = r
    ok = r["lane2"] > 0.999
    print(f"{tc:6}{r['coverage']:>10.3f}{r['precision']:>11.3f}{r['lane2']:>8.3f}   {'OK' if ok else 'CHECK'}")
n_ok = sum(v["lane2"] > 0.999 for v in l2.values())
print(f"\n{n_ok}/{len(MODELS)} pass renaming invariance (score 1.000 under a full rename)")

LANE 2 — structural comparison, renaming invariance (v2 schema)
case    coverage  precision   lane2   verdict
----------------------------------------------
1.1        1.000      1.000   1.000   OK
1.2        1.000      1.000   1.000   OK
2.1        1.000      1.000   1.000   OK
2.2        1.000      1.000   1.000   OK
3.1        1.000      1.000   1.000   OK
3.2        1.000      1.000   1.000   OK
4.1        1.000      1.000   1.000   OK
4.2        1.000      1.000   1.000   OK
4.3        1.000      1.000   1.000   OK
4.4        1.000      1.000   1.000   OK
G.1        1.000      1.000   1.000   OK
G.2        1.000      1.000   1.000   OK

12/12 pass renaming invariance (score 1.000 under a full rename)


In [9]:
# === Lane 2 error sensitivity (TC 1.1): it must PUNISH real errors, not just reward renames ===
import copy
base = MODELS["1.1"]
def Sx(m, sid): return next(s for s in m["states"] if s["id"] == sid)

renamed  = scramble(base)
prob     = copy.deepcopy(base); Sx(prob,"s0")["actions"][0]["next"]=[{"state":"s1","p":0.8},{"state":"s0","p":0.2}]
dropped  = copy.deepcopy(base); Sx(dropped,"s0")["actions"].pop(2)
wrong    = copy.deepcopy(base); Sx(wrong,"s0")["actions"][0]["next"]=[{"state":"s3","p":1.0}]
invented = copy.deepcopy(base)
invented["states"].append({"id":"s9","desc":"invented","terminal":True})
Sx(invented,"s0")["actions"].append({"by":{"robot":"call_human"},"next":[{"state":"s9","p":1.0}]})

rows=[("identical (control)",base),("fully renamed",renamed),("probability 1.0->0.8/0.2",prob),
      ("dropped an action",dropped),("wrong transition target",wrong),("invented action+state",invented)]
print(f"{'candidate model':28}{'cov':>7}{'prec':>8}{'lane2':>8}")
print("-"*51)
for n,c in rows:
    r=lane2_score(base,c,horizon=3,sa_iters=8000)
    print(f"{n:28}{r['coverage']:>7.3f}{r['precision']:>8.3f}{r['lane2']:>8.3f}")
print("\n'invented' drops PRECISION below coverage -> the signal for over-generation")
print("(the invention failure Ayman & Amna reported across 5 model families).")

candidate model                 cov    prec   lane2
---------------------------------------------------
identical (control)           1.000   1.000   1.000
fully renamed                 1.000   1.000   1.000
probability 1.0->0.8/0.2      0.972   0.972   0.972
dropped an action             0.271   0.340   0.302
wrong transition target       0.241   0.268   0.254
invented action+state         0.352   0.293   0.320

'invented' drops PRECISION below coverage -> the signal for over-generation
(the invention failure Ayman & Amna reported across 5 model families).


## Combined summary

In [ ]:
# ===  across all 12 reference models ===
print(f"{'case':6}{'Lane 1':>10}{'Lane 2 (rename-inv)':>22}")
print("-"*40)
for tc in MODELS:
    v1 = l1[tc]["verdict"]
    v2 = f"{l2[tc]['lane2']:.3f}"
    print(f"{tc:6}{v1:>10}{v2:>22}")
print("-"*40)
print(f"Lane 1: {sum(v['verdict']=='PASS' for v in l1.values())}/{len(MODELS)} pass")
print(f"Lane 2: {sum(v['lane2']>0.999 for v in l2.values())}/{len(MODELS)} rename-invariant")
print("\nLane 2 also correctly punishes real errors (see the sensitivity table above):")
print("  renamed 1.000 | prob error 0.97 | dropped action 0.30 | invented 0.32 (precision<coverage)")

case      Lane 1   Lane 2 (rename-inv)
----------------------------------------
1.1         PASS                 1.000
1.2         PASS                 1.000
2.1         PASS                 1.000
2.2         PASS                 1.000
3.1         PASS                 1.000
3.2         PASS                 1.000
4.1         PASS                 1.000
4.2         PASS                 1.000
4.3         PASS                 1.000
4.4         PASS                 1.000
G.1         PASS                 1.000
G.2         PASS                 1.000
----------------------------------------
Lane 1: 12/12 pass
Lane 2: 12/12 rename-invariant

Lane 2 also correctly punishes real errors (see the sensitivity table above):
  renamed 1.000 | prob error 0.97 | dropped action 0.30 | invented 0.32 (precision<coverage)


## Coverage across all 27 cases, and what's still planned

**Implemented and run above:** Lane 1 and Lane 2 on the 12 cases that have a reference model
(toy 1.1–4.4, game G.1, G.2).

**Not yet runnable, by design:**
- **Lane 1 on the 11 situational + 4 cultural cases**: the checks themselves need no ground truth,
  but they need a *generated model* to check, and those outputs don't exist yet. The moment Amna
  (situational) / Ani (cultural) produce outputs, this same Lane 1 runs on them unchanged.
- **Lane 2 on S/C cases**: not applicable: there is no reference model for cases with no single
  correct answer.


## Appendix: Self-test of the mechanical checks (negative testing)

The cell below takes two v2 reference models (TC 1.1 and TC 4.1), applies a set of deliberate mutations
each introducing one specific defect (bad probability sums, zero probabilities, hallucinated states, unreachable states, broken action profiles, malformed initial priors, ...) 
and confirms Lane 1 catches every one.

In [11]:
# === Self-test: deliberately broken v2 models, each must FAIL Lane 1 ===
# Proof that a PASS means something: a checker with a bug passes everything, so we
# feed it models broken one specific way each and confirm every one is caught.
import copy

def _mut(base, fn):
    m = copy.deepcopy(base); fn(m); return m

def _S(m, sid):
    return next(s for s in m["states"] if s["id"] == sid)

_single = MODELS["1.1"]   # one-player model
_multi  = MODELS["4.1"]   # two-player model

def _set_next(m, sid, i, nxt):
    _S(m, sid)["actions"][i]["next"] = nxt

MUTATIONS = [
    ("probabilities sum to 0.9",        _mut(_single, lambda m: _set_next(m,"s0",0,[{"state":"s1","p":0.9}]))),
    ("probability > 1",                 _mut(_single, lambda m: _set_next(m,"s0",2,[{"state":"s2","p":1.4}]))),
    ("zero probability (v2: not allowed)",_mut(_single, lambda m: _set_next(m,"s0",0,[{"state":"s1","p":1.0},{"state":"s0","p":0.0}]))),
    ("hallucinated successor state",    _mut(_single, lambda m: _set_next(m,"s0",0,[{"state":"s99","p":1.0}]))),
    ("undeclared player in turn",       _mut(_single, lambda m: _S(m,"s0").__setitem__("turn", _S(m,"s0")["turn"]+["ghost"]))),
    ("unreachable state",               _mut(_single, lambda m: m["states"].append({"id":"s9","desc":"orphan","terminal":True}))),
    ("terminal state with actions",     _mut(_single, lambda m: _S(m,"s1").update({"terminal":True,"turn":["robot"],
                                            "actions":[{"by":{"robot":"x"},"next":[{"state":"s1","p":1.0}]}]}))),
    ("non-terminal dead end",           _mut(_single, lambda m: (_S(m,"s3").update({"terminal":False})))),
    ("no path to terminal (self-loop)", _mut(_single, lambda m: (_S(m,"s0")["actions"].clear(),
                                            _S(m,"s0")["actions"].append({"by":{"robot":"stuck"},"next":[{"state":"s0","p":1.0}]})))),
    ("initial prior sums to 0.9",       _mut(_single, lambda m: m.__setitem__("initial",[{"state":"s0","p":0.9}]))),
    ("initial references missing state",_mut(_single, lambda m: m.__setitem__("initial",[{"state":"nope","p":1.0}]))),
    ("duplicate player ids",            _mut(_multi,  lambda m: m["players"].append({"id":m["players"][0]["id"],"type":"robot","desc":"dup"}))),
    ("missing joint action profile",    _mut(_multi,  lambda m: next(s for s in m["states"] if s.get("actions"))["actions"].pop())),
    ("action 'by' misses a turn player",_mut(_multi,  lambda m: next(s for s in m["states"] if s.get("actions"))["actions"][0]["by"].popitem())),
    ("missing top-level 'states'",      _mut(_single, lambda m: m.pop("states"))),
    ("missing top-level 'initial'",     _mut(_single, lambda m: m.pop("initial"))),
    ("players not a list",              _mut(_single, lambda m: m.__setitem__("players", "robot"))),
]

caught = 0
for name, broken in MUTATIONS:
    r = run_lane1(broken, name)
    ok = r["verdict"] == "FAIL"
    caught += ok
    print(f"  {'CAUGHT ' if ok else '!! MISSED'}  {name}")
print(f"\n{caught}/{len(MUTATIONS)} caught" +
      ("  — self-test PASSED" if caught == len(MUTATIONS) else "  — SELF-TEST FAILED"))


  CAUGHT   probabilities sum to 0.9
  CAUGHT   probability > 1
  CAUGHT   zero probability (v2: not allowed)
  CAUGHT   hallucinated successor state
  CAUGHT   undeclared player in turn
  CAUGHT   unreachable state
  CAUGHT   terminal state with actions
  CAUGHT   non-terminal dead end
  CAUGHT   no path to terminal (self-loop)
  CAUGHT   initial prior sums to 0.9
  CAUGHT   initial references missing state
  CAUGHT   duplicate player ids
  CAUGHT   missing joint action profile
  CAUGHT   action 'by' misses a turn player
  CAUGHT   missing top-level 'states'
  CAUGHT   missing top-level 'initial'
  CAUGHT   players not a list

17/17 caught  — self-test PASSED
